# Earthquake Estimator · Preprocesamiento de datos

Este cuaderno prepara los registros sísmicos del catálogo USGS para el modelado. Parte de los hallazgos del [análisis exploratorio](01_exploracion_datos.ipynb).

Cargamos `data/database.csv`, dividimos los datos en entrenamiento, prueba y validación (70/20/10), creamos la variable `peso_zona_sismica` y armamos un `Pipeline` de scikit-learn que se ajusta solo con el conjunto de entrenamiento.

## 1. Carga del dataset

Cargamos `data/database.csv` con pandas, usando la misma ruta que el cuaderno de exploración. La ruta funciona tanto si el cuaderno se ejecuta desde `notebooks/` como desde la raíz del proyecto.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "database.csv")

### Comprobación rápida de la carga

Verificamos el número de registros y columnas, y mostramos las primeras filas. El catálogo debe contener **23.412 registros y 21 columnas**.

In [ ]:
print(f"Registros: {df.shape[0]} · Columnas: {df.shape[1]}")
df.head()

## 2. Partición en entrenamiento, prueba y validación

Dividimos el catálogo completo en **70 % entrenamiento, 20 % prueba y 10 % validación** con `train_test_split`. La partición es aleatoria y usa una semilla fija (`SEMILLA = 42`) para que se pueda reproducir.

Primero separamos el 70 % de entrenamiento y después dividimos el 30 % restante en dos partes: dos tercios para prueba (20 % del total) y un tercio para validación (10 % del total).

La partición se hace **antes de ajustar cualquier transformación**. Así, el peso de zona sísmica y el preprocesamiento aprenden sus estadísticas solo con el entrenamiento, y la prueba y la validación quedan como datos que el modelo no ha visto.

> El EDA proponía una división temporal (sección 12 del cuaderno 01). Aquí usamos una partición aleatoria con semilla fija, como se decidió para esta etapa. Tampoco aplicamos todavía el filtro `Type == "Earthquake"`: trabajamos con los 23.412 registros de `df`.

In [ ]:
from sklearn.model_selection import train_test_split

SEMILLA = 42
variable_objetivo = "Magnitude"

X = df.drop(columns=variable_objetivo)
y = df[variable_objetivo]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEMILLA
)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=1 / 3, random_state=SEMILLA
)

conjuntos = {
    "Entrenamiento": (X_train, y_train),
    "Prueba": (X_test, y_test),
    "Validación": (X_val, y_val),
}
resumen_particion = pd.DataFrame(
    {
        "registros": [len(Xc) for Xc, _ in conjuntos.values()],
        "porcentaje": [round(len(Xc) / len(df) * 100, 2) for Xc, _ in conjuntos.values()],
    },
    index=list(conjuntos),
)
print(resumen_particion.to_string())
print(f"\nTotal asignado: {resumen_particion['registros'].sum()} de {len(df)}")

### Resultado de la partición

Asignamos **16.388 registros (70 %)** a entrenamiento, **4.682 (20 %)** a prueba y **2.342 (10 %)** a validación. La suma coincide con los 23.412 registros del catálogo.

## 3. Feature engineering: peso de zona sísmica

Creamos una nueva variable de entrada, `peso_zona_sismica`, con un valor de **1 a 100** que indica qué tan propensa es una ubicación (`Latitude`, `Longitude`) a tener terremotos fuertes. No es la variable objetivo: `Magnitude` sigue siendo lo que el modelo estimará.

La lógica está en el transformador `PesoZonaSismica` del módulo [`src/zona_sismica.py`](../src/zona_sismica.py). Es un transformador de scikit-learn (`BaseEstimator` + `TransformerMixin`), por lo que puede ir dentro del `Pipeline` y reutilizarse más adelante en la aplicación.

### Cómo se calcula el peso

**Cuadrícula.** Dividimos el mapa del mundo en casillas de **2° × 2°** (≈ 222 km de lado, parámetro `cell_size_deg=2`). Cada punto cae en una casilla de 90 filas × 180 columnas.

**Componente histórico (0-1)**, aprendido en `fit()` solo con el entrenamiento:
- *Actividad*: número de eventos de la casilla, en escala logarítmica y dividido entre el máximo.
- *Proporción de terremotos fuertes*: fracción de eventos con `Magnitude >= 6.5`. Como usa la variable objetivo, la suavizamos hacia la proporción global del entrenamiento sumando 20 eventos "virtuales" con esa proporción (`prior_strength=20`). Así, una casilla con un solo evento fuerte no recibe una proporción del 100 %.
- *Vecinas*: antes de calcular ambas medidas, a cada casilla le sumamos los conteos de sus 8 vecinas con peso 0,5 (`neighbor_weight`). Una casilla vacía junto a una zona activa no cae al mínimo.
- El componente histórico es el promedio de la actividad y la proporción de fuertes.

**Componente tectónico (0-1)**, con datos externos: usamos los límites de placas del modelo PB2002 (Bird, 2002), guardados en `data/PB2002_steps.csv`. Para cada punto calculamos la distancia (haversine) al límite más cercano de cada tipo y la convertimos en puntaje con `peso_tipo × exp(-distancia / 300 km)`. Nos quedamos con el mayor puntaje:

| Tipo de límite | Clases PB2002 | Peso |
|---|---|---|
| Convergente / subducción | `SUB`, `OCB`, `CCB` | 1,0 |
| Transformante | `OTF`, `CTF` | 0,6 |
| Divergente | `OSR`, `CRB` | 0,3 |

**Combinación y escala.** `bruto = 0,6 × histórico + 0,4 × tectónico`. En `fit()` calculamos los percentiles 1 y 99 de `bruto` en el entrenamiento y los usamos para llevar el valor a la escala 1-100. Los valores fuera de ese rango se recortan a 1 o a 100.

### Decisión sobre los pesos de la combinación (0,6 / 0,4)

Damos más peso al componente histórico porque resume lo que ocurrió en el propio catálogo, casilla por casilla. El componente tectónico recibe 0,4 porque aporta información física aunque la casilla tenga pocos eventos, y porque no depende de la magnitud, lo que limita cuánto del peso se parece a una codificación de la variable objetivo. Son pesos iniciales: se pueden ajustar con `hist_weight` y comparar con el conjunto de validación cuando entrenemos modelos.

### Cómo evitamos la fuga de información (target leakage)

- El transformador solo aprende en `fit()`, que el `Pipeline` llama con el entrenamiento. Prueba, validación y la futura aplicación usan `transform()`, que no recibe `Magnitude`.
- En el entrenamiento, el componente histórico de cada fila se calcula **fuera de muestra** con 5 particiones (`n_splits=5`): cada fila recibe el valor de una cuadrícula ajustada sin su propia magnitud. Es la misma idea que usa `TargetEncoder` de scikit-learn. Sin esto, el peso de entrenamiento incluiría la magnitud de la propia fila y el modelo lo sobrevaloraría.
- Las casillas sin eventos de entrenamiento reciben la proporción global y el componente tectónico, por lo que cualquier punto del mapa tiene un peso.

In [ ]:
import sys

sys.path.insert(0, str(project_root / "src"))

from zona_sismica import PesoZonaSismica

## 4. Pipeline de preprocesamiento

El `Pipeline` tiene dos pasos:

1. `peso_zona`: el transformador `PesoZonaSismica`, que agrega la columna `peso_zona_sismica`.
2. `preprocesamiento`: un `ColumnTransformer` que selecciona las entradas del modelo y descarta las demás columnas.

Según el EDA, las entradas iniciales son `Latitude`, `Longitude` y `Depth`; a ellas sumamos `peso_zona_sismica`. Las cuatro son numéricas, así que no necesitamos codificar categorías. Ninguna tiene faltantes en el catálogo, pero incluimos `SimpleImputer` con la mediana para que el pipeline también funcione si la aplicación recibe un dato incompleto. Después estandarizamos con `StandardScaler`. Los árboles no lo necesitan, pero deja las entradas listas para modelos sensibles a la escala.

Ajustamos el pipeline **solo con el entrenamiento** (`fit_transform(X_train, y_train)`) y aplicamos `transform` a prueba y validación.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

columnas_numericas = ["Latitude", "Longitude", "Depth", "peso_zona_sismica"]

preprocesamiento = ColumnTransformer(
    [
        (
            "numericas",
            Pipeline(
                [
                    ("imputar", SimpleImputer(strategy="median")),
                    ("escalar", StandardScaler()),
                ]
            ),
            columnas_numericas,
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

pipeline = Pipeline(
    [
        ("peso_zona", PesoZonaSismica(cell_size_deg=2, random_state=SEMILLA)),
        ("preprocesamiento", preprocesamiento),
    ]
)

X_train_prep = pipeline.fit_transform(X_train, y_train)
X_test_prep = pipeline.transform(X_test)
X_val_prep = pipeline.transform(X_val)

print(f"Entrenamiento: {X_train_prep.shape}")
print(f"Prueba: {X_test_prep.shape}")
print(f"Validación: {X_val_prep.shape}")
X_train_prep.head()

### Formas de los conjuntos transformados

Los tres conjuntos quedan con **4 columnas** (`Latitude`, `Longitude`, `Depth` y `peso_zona_sismica`, estandarizadas): entrenamiento (16.388, 4), prueba (4.682, 4) y validación (2.342, 4). El resto de columnas del CSV se descarta en el `ColumnTransformer`.

## 5. Revisión del peso de zona sísmica

### Comprobación de que el ajuste usa solo el entrenamiento

Verificamos que los tres conjuntos no comparten filas y que las estadísticas guardadas por el transformador (número de eventos, terremotos fuertes y proporción global) coinciden exactamente con el entrenamiento.

In [ ]:
zona = pipeline.named_steps["peso_zona"]
umbral = zona.strong_threshold

indices = {nombre: set(Xc.index) for nombre, (Xc, _) in conjuntos.items()}
assert indices["Entrenamiento"].isdisjoint(indices["Prueba"])
assert indices["Entrenamiento"].isdisjoint(indices["Validación"])
assert indices["Prueba"].isdisjoint(indices["Validación"])

assert zona.n_eventos_fit_ == len(X_train)
assert zona.conteo_.sum() == len(X_train)
assert zona.fuertes_.sum() == (y_train >= umbral).sum()
assert np.isclose(zona.prior_fuerte_, (y_train >= umbral).mean())

print(f"Eventos usados en fit: {zona.n_eventos_fit_} (entrenamiento: {len(X_train)})")
print(f"Terremotos fuertes (M >= {umbral}) en fit: {int(zona.fuertes_.sum())}")
print(f"Proporción global de fuertes (entrenamiento): {zona.prior_fuerte_:.4f}")
print(f"Proporción de fuertes en prueba (no usada): {(y_test >= umbral).mean():.4f}")
print(f"Percentiles 1 y 99 del puntaje bruto: {zona.percentiles_[0]:.4f} y {zona.percentiles_[1]:.4f}")
print("Sin filas compartidas entre conjuntos y estadísticas calculadas solo con entrenamiento.")

### Resultado de la comprobación

El transformador usó los **16.388 eventos de entrenamiento**, de los cuales **1.600** tienen magnitud de 6,5 o más. Su proporción global de fuertes es **0,0976**, la del entrenamiento; la de prueba (0,0946) no interviene en el ajuste. Los conjuntos no comparten filas.

### Resumen del peso en cada conjunto

Para el entrenamiento usamos el peso fuera de muestra que calculó `fit` (`peso_entrenamiento_`). Para prueba y validación usamos `transform`. Comprobamos que todos los valores quedan entre 1 y 100.

In [ ]:
pesos = {
    "Entrenamiento": pd.Series(zona.peso_entrenamiento_, index=X_train.index),
    "Prueba": zona.transform(X_test)["peso_zona_sismica"],
    "Validación": zona.transform(X_val)["peso_zona_sismica"],
}
for nombre, peso in pesos.items():
    assert peso.between(1, 100).all(), nombre

resumen_peso = pd.DataFrame({nombre: peso.describe() for nombre, peso in pesos.items()}).T
print(resumen_peso.round(2).to_string())

### Interpretación del resumen

Los tres conjuntos tienen valores entre **1 y 100** y distribuciones muy parecidas: la media es **62,70** en entrenamiento, **62,31** en prueba y **62,16** en validación, y la mediana está cerca de **66** en los tres. La mayoría de los eventos tiene pesos altos porque el catálogo ya se concentra en zonas sísmicas activas.

### Relación del peso con la magnitud

Calculamos las correlaciones de Pearson y Spearman entre el peso y `Magnitude`. En el entrenamiento usamos el peso fuera de muestra; en prueba, el peso de `transform`. Si los valores son parecidos, el peso no está memorizando la magnitud de las filas de entrenamiento.

In [ ]:
objetivos = {"Entrenamiento": y_train, "Prueba": y_test, "Validación": y_val}
correlaciones = pd.DataFrame(
    {
        nombre: {
            "pearson": pesos[nombre].corr(objetivos[nombre]),
            "spearman": pesos[nombre].corr(objetivos[nombre], method="spearman"),
        }
        for nombre in pesos
    }
).T
print(correlaciones.round(3).to_string())

fuertes_por_rango = (
    pd.DataFrame({"peso": pesos["Entrenamiento"], "fuerte": y_train >= umbral})
    .assign(rango=lambda d: pd.cut(d["peso"], bins=[0, 20, 40, 60, 80, 100]))
    .groupby("rango", observed=True)["fuerte"]
    .agg(registros="size", proporcion_fuertes="mean")
)
print("\nProporción de terremotos fuertes por rango de peso (entrenamiento):")
print(fuertes_por_rango.round(3).to_string())

### Interpretación de la relación con la magnitud

Las correlaciones son débiles y positivas: Pearson **0,037** en entrenamiento, **0,020** en prueba y **0,061** en validación. Que el entrenamiento no tenga una correlación mucho mayor que prueba y validación indica que el cálculo fuera de muestra evita que el peso memorice la magnitud de sus propias filas.

La proporción de terremotos fuertes es mayor en el rango 80-100 (**11,1 %**) que en el rango 20-40 (**8,1 %**). La relación es pequeña porque el catálogo solo incluye magnitudes de 5,5 o más. El aporte real del peso se medirá al comparar modelos con y sin esta variable.

### Peso en lugares de referencia

Aplicamos el transformador a ciudades conocidas. Esperamos pesos altos en el Cinturón de Fuego del Pacífico (Japón, Chile, Indonesia, Alaska) y bajos en interiores de placa estables (Brasil, Rusia europea, Australia). Estos puntos no están en el catálogo, lo que también comprueba que `transform` funciona con ubicaciones nuevas.

In [ ]:
referencias = pd.DataFrame(
    {
        "lugar": [
            "Tokio", "Santiago de Chile", "Yakarta", "Anchorage", "Lima",
            "Ciudad de México", "Bogotá", "Madrid", "Brasilia", "Moscú", "Sídney",
        ],
        "Latitude": [35.68, -33.45, -6.21, 61.22, -12.05, 19.43, 4.71, 40.42, -15.79, 55.76, -33.87],
        "Longitude": [139.69, -70.67, 106.85, -149.90, -77.04, -99.13, -74.07, -3.70, -47.88, 37.62, 151.21],
    }
)
peso_referencias = pd.concat(
    [referencias, zona.componentes(referencias).round(3)], axis=1
).assign(peso_zona_sismica=zona.transform(referencias)["peso_zona_sismica"].round(1))
print(peso_referencias.sort_values("peso_zona_sismica", ascending=False).to_string(index=False))

### Interpretación de los lugares de referencia

Las ciudades cercanas a zonas de subducción del Pacífico y de los Andes reciben los pesos más altos: Tokio (**71,0**), Bogotá (**63,4**), Santiago de Chile (**59,5**) y Lima (**49,2**). Brasilia, Moscú y Sídney, en interiores de placa, quedan en el mínimo (**1,0**), y Madrid casi en el mínimo (**1,9**).

Yakarta (**33,6**) y Anchorage (**28,0**) quedan en valores intermedios: están a cierta distancia de la fosa de subducción y su casilla de 2° tiene menos eventos que las casillas sobre la fosa.

### Mapa del peso en el entrenamiento

Cada punto es un evento de entrenamiento y su color indica el peso fuera de muestra.

In [ ]:
import plotly.express as px

mapa_peso = X_train[["Latitude", "Longitude"]].assign(
    peso_zona_sismica=pesos["Entrenamiento"], Magnitude=y_train
)
fig_peso = px.scatter_geo(
    mapa_peso,
    lat="Latitude",
    lon="Longitude",
    color="peso_zona_sismica",
    hover_data={"Magnitude": ":.1f", "peso_zona_sismica": ":.1f"},
    color_continuous_scale="Inferno",
    range_color=(1, 100),
    projection="natural earth",
    opacity=0.6,
    title="Peso de zona sísmica de los eventos de entrenamiento",
    labels={"peso_zona_sismica": "Peso (1-100)", "Magnitude": "Magnitud"},
)
fig_peso.update_traces(marker=dict(size=3))
fig_peso.update_geos(showland=True, landcolor="lightgray", showocean=True, oceancolor="aliceblue")
fig_peso.update_layout(height=600)
fig_peso.show()